# Compute explanation times

In [3]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


## Relevant libraries

In [21]:
from functools import partial
import time
import numpy as np
import pandas as pd

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.reconstruction import ReconstructionAnomalyScore
from anomaly.utils import FilterParameters, ReconstructionParameters

from astroExplain.spectra.explainer import LimeSpectraExplainer
from astroExplain.spectra.segment import SpectraSegmentation
from autoencoders.ae import AutoEncoder
from sdss.metadata import MetaData

meta = MetaData()

## Custom functions

# Data ingestion

In [5]:
data_dir = "/home/elom/spectra"
model_dir = "/home/elom/models"
bin_id = "bin_03"
explanations_dir = f"{model_dir}/{bin_id}/explanation"
paper_figures_dir = "/home/elom/phd/00_paper_explain-me-why/sections/figures/"
meta_data_df = pd.read_csv(
    f"{data_dir}/0_01_z_0_5_4_0_snr_inf.csv.gz",
    index_col="specobjid",
)
wave = np.load(f"{data_dir}/wave_spectra_imputed.npy")

spectra = np.load(
    f"{data_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

idx_id = np.load(
    f"{data_dir}/{bin_id}/{bin_id}_index_specobjid.npy"
)

## Load model

In [7]:
ae_model = AutoEncoder(
    reload=True,
    reload_from=f"{model_dir}/{bin_id}",
)

# Explanation Hyperparameters

In [33]:
# config file
score_config = {
    "metric": "mse",
    "velocity": 0,
    # if reconstruction
    "relative": False,
    "percentage": 100,
    "epsilon": 0.001,
    "lines": list(GALAXY_LINES.keys()),
}

lime_config = {
    "segmentation": "uniform",
    "number_segments": 64,
    "number_samples": 5000,
    "batch_size": 100,
    "progress_bar": False,
    "distance_metric": "cosine",
    "number_features": 5_000,
}

fudge_config = {
    "kind_of_fudge": "scale",
    # scale
    "scale_factor": 0.9,
    # control-noise
    "same_noise": True,
    "kernel_size": 3,
    "sigma": 0,
}

# Define anomaly scoring function

In [18]:
anomaly = ReconstructionAnomalyScore(
    # reconstruct_function
    ae_model.reconstruct,
    filter_parameters=FilterParameters(
        wave=wave,
        lines=score_config["lines"],
        velocity_filter=score_config["velocity"],
    ),
    reconstruction_parameters=ReconstructionParameters(
        percentage=score_config["percentage"],
        relative=score_config["relative"],
        epsilon=score_config["epsilon"],
    ),
)

anomaly_score_function = partial(
    anomaly.score, metric=score_config["metric"]
)
# Set explainer instance
print("Set explainer and Get explanations", end="\n")
explainer = LimeSpectraExplainer(random_state=0)

segmentation_fn = None

if lime_config["segmentation"] == "kmeans":

    segmentation_fn = SpectraSegmentation().kmeans

elif lime_config["segmentation"] == "uniform":

    segmentation_fn = SpectraSegmentation().uniform

segmentation_fn = partial(
    segmentation_fn, number_segments=lime_config["number_segments"]
)


Set explainer and Get explanations


# Measure explanation times

In [25]:
# Get explanation for a specific specobjid
strong_emission_lines = {
    'specobjid': 3240467000396376064,
    'metric': 'mse_noRel100',
    'explanation': 'uniform_128_scale_0.9',
}

specobjid = strong_emission_lines["specobjid"]
idx_spectrum = specobjid_to_idx(specobjid, idx_id)
spectrum = spectra[idx_spectrum]
spectrum = spectrum[np.newaxis, :]


In [34]:
n_samples = [10, 100, 1_000, 5_000, 10_000]
speeds = []

for n in n_samples:

    lime_config["number_samples"] = n

    t_start = time.perf_counter()

    (
        explanation,
        ret_exp_score,
        ret_exp_local_pred
    ) = explainer.explain_instance(
        spectrum=spectrum,
        classifier_fn=anomaly_score_function,
        segmentation_fn=segmentation_fn,
        fudge_parameters=fudge_config,
        explainer_parameters=lime_config,
    )
    t_end = time.perf_counter()
    speeds.append(t_end - t_start)
    print(
        f"Explained {n} samples in {t_end - t_start:.2f} seconds"
    )
speeds

Explained 10 samples in 0.08 seconds
Explained 100 samples in 0.14 seconds
Explained 1000 samples in 1.19 seconds
Explained 5000 samples in 6.51 seconds
Explained 10000 samples in 12.56 seconds


[0.0755032220040448,
 0.13988797798810992,
 1.1924579799961066,
 6.507999982000911,
 12.558957809000276]